Pre-Code

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional

import sympy as sp

In [ ]:
__all__ = [
    "Equation"
    "translate"
    "preprocess"
    "tokenize"
    "Parser"
    "name_to_latex"
    "latex_to_name"
    "LatextParseError"
]

Defining Error

In [ ]:
class LatexParseError(ValueError):
    """Raised when a LaTeX fragment cannot be parsed into a Python expression."""

Symbol / Function / Decorator Mapping Dictionary (Hard-Coded - Must update if necessary)

In [ ]:
# Greek letters -> plain-ASCII SymPy symbol names.  
_GREEK = {
    "alpha": "alpha", 
    "beta": "beta", 
    "gamma": "gamma", 
    "delta": "delta",
    "epsilon": "epsilon", 
    "varepsilon": "epsilon", 
    "zeta": "zeta", 
    "eta": "eta",
    "theta": "theta", 
    "vartheta": "theta", 
    "iota": "iota", 
    "kappa": "kappa",
    "lambda": "lamda", 
    "mu": "mu", 
    "nu": "nu", 
    "xi": "xi", 
    "pi": "pi",
    "rho": "rho", 
    "varrho": "rho", 
    "sigma": "sigma", 
    "varsigma": "sigma",
    "tau": "tau",
    "upsilon": "upsilon", 
    "phi": "phi", 
    "varphi": "phi",
    "chi": "chi", 
    "psi": "psi", 
    "omega": "omega",
    "Gamma": "Gamma", 
    "Delta": "Delta", 
    "Theta": "Theta", 
    "Lambda": "Lamda",  # `lambda` is mapped to ``lamda`` (the same spelling SymPy uses). lambda is a reserved keyword in Python, so we cannot use it as a symbol name.
    "Xi": "Xi", 
    "Pi": "Pi", 
    "Sigma": "Sigma", 
    "Upsilon": "Upsilon",
    "Phi": "Phi", 
    "Psi": "Psi", 
    "Omega": "Omega",
}

# LaTeX standard functions -> SymPy callables.
_STDFUNCTIONS: Dict[str, Callable] = {
    "exp": sp.exp,
    "log": sp.log,
    "ln": sp.log,
    "sqrt": sp.sqrt,
    "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
    "sec": sp.sec, "csc": sp.csc, "cot": sp.cot,
    "arcsin": sp.asin, "arccos": sp.acos, "arctan": sp.atan,
    "sinh": sp.sinh, "cosh": sp.cosh, "tanh": sp.tanh,
    "abs": sp.Abs,
}

# Decorators are commands that decorate a maths symbol.  
# Several LaTeX spellings share a suffix on purpose (``\bar`` and ``\overline`` both read as "bar")
# An empty suffix marks a purely cosmetic wrapper, so will not affect the generated name (``\mathrm{t}`` -> ``t``).
_DECORATORS = {
    "bar": "bar", "overline": "bar", "hat": "hat", "widehat": "hat",
    "tilde": "tilde", "widetilde": "tilde", "vec": "vec", "dot": "dot",
    "ddot": "ddot", "overrightarrow": "vec", "mathbf": "", "boldsymbol": "",
    "mathrm": "", "text": "", "operatorname": "",
}

Tokeniser

In [ ]:
# Below are operators with no algebraic Python form here (e.g., integrals, summations, limits, partial derivatives).
# Current tokeniser is unable to translate them. For examples, ``\int_{0}^{t}`` reads as ``int_0**t``.
# So an equation that uses one is reported as untranslatable.

# Add more if you find them

_UNSUPPORTED_OPERATORS = {
    "int": "an integral (\\int)",
    "iint": "a double integral (\\iint)",
    "iiint": "a triple integral (\\iiint)",
    "oint": "a contour integral (\\oint)",
    "sum": "a summation (\\sum)",
    "prod": "a product (\\prod)",
    "lim": "a limit (\\lim)",
    "partial": "a partial derivative (\\partial)",
}
